# Notebook 13 — Manuscript-Ready Exploratory Analysis, Tables, Figures, and Reporting Checklist

**Project:** Machine Learning-Based Prediction of Parkinson’s Disease Progression Using PPMI Data  
**Stage:** Final reporting preparation after model comparison  
**Purpose:** Generate manuscript-ready descriptive tables, performance tables, figures, reporting checklist, and interpretation summaries without training new models.

> This notebook does **not** tune models and does **not** re-use the held-out test set for additional optimization. It only consolidates prior outputs into transparent manuscript-ready materials.

## Objective

1. Consolidate cohort, outcome, feature-set, and model performance outputs from previous notebooks.  
2. Generate manuscript-ready tables and figures for reporting.  
3. Create a transparent final decision summary and limitations table.  
4. Confirm whether the project is ready to move into manuscript drafting.

## Scientific Background

The preceding notebooks constructed a prediction task for Parkinson’s disease motor progression using longitudinal MDS-UPDRS Part III change. Clinical predictors were evaluated first, followed by multimodal additions from DaTSCAN/SBR and SAA biomarker features. The final reporting emphasis is not a high-performance deployable clinical model, but a reproducible benchmark comparing clinical, imaging-derived, and biomarker-enhanced feature sets for predicting rapid motor progression.

## Dataset Verification

This notebook expects outputs from:

- Notebook 02 — Cohort Definition and Outcome Construction  
- Notebook 04 — Preprocessing Pipeline and Train/Test Split  
- Notebook 08 — Multimodal Feature Integration and Preprocessing  
- Notebook 10 — Biomarker Data Verification and Feature Integration  
- Notebook 12 — Final Model Selection and Manuscript-Ready Performance Summary  

No participant-level raw PPMI data is exported outside the local project directory.

In [ ]:
# ============================================================
# 01. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# 02. Imports and project paths
# ============================================================

from pathlib import Path
import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42

PROJECT_DIR = Path("/content/drive/MyDrive/PPMI_PD_Progression")

NB02_DIR = PROJECT_DIR / "outputs" / "notebook_02_cohort_outcome"
NB04_DIR = PROJECT_DIR / "outputs" / "notebook_04_preprocessing"
NB08_DIR = PROJECT_DIR / "outputs" / "notebook_08_multimodal_preprocessing"
NB10_DIR = PROJECT_DIR / "outputs" / "notebook_10_biomarker_feature_integration"
NB12_DIR = PROJECT_DIR / "outputs" / "notebook_12_final_model_summary"

OUT_DIR = PROJECT_DIR / "outputs" / "notebook_13_manuscript_ready_reporting"
FIG_DIR = OUT_DIR / "figures"
TABLE_DIR = OUT_DIR / "tables"

for d in [OUT_DIR, FIG_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("OUT_DIR:", OUT_DIR)

expected_dirs = {
    "Notebook 02": NB02_DIR,
    "Notebook 04": NB04_DIR,
    "Notebook 08": NB08_DIR,
    "Notebook 10": NB10_DIR,
    "Notebook 12": NB12_DIR,
}

for name, path in expected_dirs.items():
    print(f"{name}: {path} | exists={path.exists()}")

In [ ]:
# ============================================================
# 03. Helper functions
# ============================================================

def find_file(directory, exact_name=None, contains=None, suffix=".csv"):
    directory = Path(directory)
    if not directory.exists():
        return None

    candidates = []
    for p in directory.rglob(f"*{suffix}"):
        if exact_name and p.name == exact_name:
            return p
        if contains and contains.lower() in p.name.lower():
            candidates.append(p)

    if candidates:
        return sorted(candidates)[0]
    return None


def read_csv_optional(path, label):
    if path is None or not Path(path).exists():
        print(f"[MISSING] {label}")
        return pd.DataFrame()
    df = pd.read_csv(path)
    print(f"[LOADED] {label}: {Path(path).name} | shape={df.shape}")
    return df


def save_table(df, filename):
    path = TABLE_DIR / filename
    df.to_csv(path, index=False)
    print("Saved:", path)
    return path


def safe_round_df(df, digits=4):
    out = df.copy()
    for col in out.select_dtypes(include=[np.number]).columns:
        out[col] = out[col].round(digits)
    return out


def pct(x, total):
    if total == 0 or pd.isna(total):
        return np.nan
    return 100 * x / total

In [ ]:
# ============================================================
# 04. Load previous notebook outputs
# ============================================================

cohort_path = find_file(NB02_DIR, exact_name="11_primary_analytic_cohort_recommended_window.csv")
outcome_summary_path = find_file(NB02_DIR, exact_name="07_candidate_outcome_summary.csv")
split_summary_path = find_file(NB04_DIR, exact_name="01_train_test_split_summary.csv")
feature_type_path = find_file(NB04_DIR, exact_name="03_feature_type_dictionary.csv")
multimodal_manifest_path = find_file(NB08_DIR, exact_name="11_feature_set_manifest.csv")
biomarker_manifest_path = find_file(NB10_DIR, exact_name="16_feature_set_manifest_with_biomarkers.csv")

final_perf_path = find_file(NB12_DIR, exact_name="05_manuscript_ready_model_performance_table.csv")
incremental_path = find_file(NB12_DIR, exact_name="02_incremental_performance_summary.csv")
decision_path = find_file(NB12_DIR, exact_name="04_final_model_selection_decision_table.csv")
consolidated_path = find_file(NB12_DIR, exact_name="01_consolidated_test_performance_all_models.csv")
top_models_path = find_file(NB12_DIR, exact_name="03_top_models_by_metric.csv")

cohort = read_csv_optional(cohort_path, "Primary analytic cohort")
outcome_summary = read_csv_optional(outcome_summary_path, "Candidate outcome summary")
split_summary = read_csv_optional(split_summary_path, "Train/test split summary")
feature_types = read_csv_optional(feature_type_path, "Feature type dictionary")
multimodal_manifest = read_csv_optional(multimodal_manifest_path, "Feature set manifest")
biomarker_manifest = read_csv_optional(biomarker_manifest_path, "Feature set manifest with biomarkers")

final_perf = read_csv_optional(final_perf_path, "Manuscript performance table")
incremental = read_csv_optional(incremental_path, "Incremental performance summary")
decision = read_csv_optional(decision_path, "Final model decision table")
consolidated = read_csv_optional(consolidated_path, "Consolidated test performance")
top_models = read_csv_optional(top_models_path, "Top models by metric")

## Code — Manuscript Tables

In [ ]:
# ============================================================
# 05. Table 1: Analytic cohort and outcome summary
# ============================================================

summary_rows = []

if not cohort.empty:
    n_total = cohort["PATNO"].nunique() if "PATNO" in cohort.columns else len(cohort)
    summary_rows.append({"domain": "Analytic cohort", "metric": "Participants", "value": n_total})

    if "rapid_progression_q75" in cohort.columns:
        n_pos = int(cohort["rapid_progression_q75"].sum())
        summary_rows.append({"domain": "Outcome", "metric": "Rapid progressors, n", "value": n_pos})
        summary_rows.append({"domain": "Outcome", "metric": "Rapid progressors, %", "value": round(pct(n_pos, n_total), 2)})

    for col in ["baseline_NP3TOT", "followup_NP3TOT", "delta_NP3TOT", "annualized_delta_NP3TOT"]:
        if col in cohort.columns:
            summary_rows.extend([
                {"domain": "Outcome score", "metric": f"{col}, mean", "value": round(cohort[col].mean(), 3)},
                {"domain": "Outcome score", "metric": f"{col}, median", "value": round(cohort[col].median(), 3)},
                {"domain": "Outcome score", "metric": f"{col}, IQR", "value": round(cohort[col].quantile(0.75) - cohort[col].quantile(0.25), 3)}
            ])

    if "followup_event" in cohort.columns:
        vals = cohort["followup_event"].dropna().unique()
        summary_rows.append({"domain": "Follow-up", "metric": "Primary follow-up event(s)", "value": "; ".join(map(str, vals))})

cohort_outcome_table = pd.DataFrame(summary_rows)
save_table(cohort_outcome_table, "01_table1_cohort_and_outcome_summary.csv")
cohort_outcome_table

In [ ]:
# ============================================================
# 06. Table 2: Train/test split and feature set summary
# ============================================================

split_table = split_summary.copy()
if not split_table.empty:
    save_table(split_table, "02_train_test_split_summary.csv")

feature_summary_rows = []

if not feature_types.empty and "feature_type" in feature_types.columns:
    ft_counts = feature_types["feature_type"].value_counts(dropna=False).reset_index()
    ft_counts.columns = ["feature_type", "n_raw_predictors"]
    for _, row in ft_counts.iterrows():
        feature_summary_rows.append({
            "feature_set": "Clinical-only raw predictors",
            "feature_type": row["feature_type"],
            "n_features": row["n_raw_predictors"]
        })

if not multimodal_manifest.empty:
    feature_summary_rows.append({
        "feature_set": "Clinical + DaTSCAN/SBR",
        "feature_type": "Processed feature-set manifest rows",
        "n_features": len(multimodal_manifest)
    })

if not biomarker_manifest.empty:
    feature_summary_rows.append({
        "feature_set": "Clinical + DaTSCAN/SBR + SAA biomarkers",
        "feature_type": "Processed feature-set manifest rows",
        "n_features": len(biomarker_manifest)
    })

feature_summary = pd.DataFrame(feature_summary_rows)
save_table(feature_summary, "03_feature_set_summary.csv")
feature_summary

In [ ]:
# ============================================================
# 07. Table 3: Manuscript-ready model performance
# ============================================================

if final_perf.empty:
    manuscript_perf = pd.DataFrame()
else:
    keep_cols = [
        "analysis_stage", "feature_set", "model", "roc_auc", "pr_auc",
        "balanced_accuracy", "sensitivity", "specificity", "precision",
        "f1", "threshold", "n_test", "n_predictors"
    ]
    keep_cols = [c for c in keep_cols if c in final_perf.columns]
    manuscript_perf = final_perf[keep_cols].copy()
    manuscript_perf = safe_round_df(manuscript_perf, 4)

    save_table(manuscript_perf, "04_manuscript_ready_model_performance_table.csv")

manuscript_perf.head(20)

In [ ]:
# ============================================================
# 08. Table 4: Incremental value summary
# ============================================================

if incremental.empty:
    incremental_clean = pd.DataFrame()
else:
    incremental_clean = incremental.copy()
    incremental_clean = safe_round_df(incremental_clean, 4)
    save_table(incremental_clean, "05_incremental_value_summary.csv")

incremental_clean

In [ ]:
# ============================================================
# 09. Table 5: Final model selection and interpretation decision
# ============================================================

if decision.empty:
    decision_table = pd.DataFrame()
else:
    decision_table = decision.copy()
    save_table(decision_table, "06_final_model_selection_decision_table.csv")

decision_table

## Code — Manuscript Figures

In [ ]:
# ============================================================
# 10. Figure 1: Outcome distribution
# ============================================================

if not cohort.empty and "annualized_delta_NP3TOT" in cohort.columns:
    plt.figure(figsize=(7, 5))
    plt.hist(cohort["annualized_delta_NP3TOT"].dropna(), bins=30)
    plt.axvline(cohort["annualized_delta_NP3TOT"].quantile(0.75), linestyle="--")
    plt.xlabel("Annualized change in MDS-UPDRS Part III")
    plt.ylabel("Number of participants")
    plt.title("Distribution of annualized motor progression")
    plt.tight_layout()
    fig_path = FIG_DIR / "figure_01_annualized_delta_NP3TOT_distribution.png"
    plt.savefig(fig_path, dpi=300)
    plt.show()
    print("Saved:", fig_path)
else:
    print("Skipped: annualized_delta_NP3TOT not available.")

In [ ]:
# ============================================================
# 11. Figure 2: Top held-out ROC-AUC models
# ============================================================

if not final_perf.empty and "roc_auc" in final_perf.columns:
    plot_df = final_perf.dropna(subset=["roc_auc"]).copy()
    plot_df = plot_df.sort_values("roc_auc", ascending=False).head(10)
    plot_df["label"] = plot_df["model"].astype(str)
    if "feature_set" in plot_df.columns:
        plot_df["label"] = plot_df["feature_set"].fillna("clinical_only").astype(str) + "\n" + plot_df["model"].astype(str)

    plt.figure(figsize=(8, 6))
    plt.barh(plot_df["label"][::-1], plot_df["roc_auc"][::-1])
    plt.xlabel("Held-out ROC-AUC")
    plt.title("Top models by held-out ROC-AUC")
    plt.xlim(0.45, max(0.70, plot_df["roc_auc"].max() + 0.03))
    plt.tight_layout()
    fig_path = FIG_DIR / "figure_02_top_models_by_roc_auc.png"
    plt.savefig(fig_path, dpi=300)
    plt.show()
    print("Saved:", fig_path)
else:
    print("Skipped: final performance table not available.")

In [ ]:
# ============================================================
# 12. Figure 3: Incremental change in performance
# ============================================================

rows = []
if not incremental.empty:
    # DaTSCAN summary row style
    for _, r in incremental.iterrows():
        label = r.get("increment_type", r.get("comparison", "increment"))
        if pd.notna(r.get("delta_roc_auc", np.nan)):
            rows.append({"increment": label, "metric": "ROC-AUC", "delta": r["delta_roc_auc"]})
        if pd.notna(r.get("delta_pr_auc", np.nan)):
            rows.append({"increment": label, "metric": "PR-AUC", "delta": r["delta_pr_auc"]})
        if pd.notna(r.get("delta_balanced_accuracy", np.nan)):
            rows.append({"increment": label, "metric": "Balanced accuracy", "delta": r["delta_balanced_accuracy"]})
        if pd.notna(r.get("delta_test_roc_auc", np.nan)):
            rows.append({"increment": label, "metric": "ROC-AUC", "delta": r["delta_test_roc_auc"]})
        if pd.notna(r.get("delta_test_pr_auc", np.nan)):
            rows.append({"increment": label, "metric": "PR-AUC", "delta": r["delta_test_pr_auc"]})
        if pd.notna(r.get("delta_test_balanced_accuracy", np.nan)):
            rows.append({"increment": label, "metric": "Balanced accuracy", "delta": r["delta_test_balanced_accuracy"]})

increment_plot = pd.DataFrame(rows)

if not increment_plot.empty:
    increment_plot["plot_label"] = increment_plot["increment"].astype(str) + "\n" + increment_plot["metric"].astype(str)
    plt.figure(figsize=(8, 5))
    plt.barh(increment_plot["plot_label"][::-1], increment_plot["delta"][::-1])
    plt.axvline(0, linestyle="--")
    plt.xlabel("Incremental change in held-out performance")
    plt.title("Incremental value of multimodal features")
    plt.tight_layout()
    fig_path = FIG_DIR / "figure_03_incremental_performance_changes.png"
    plt.savefig(fig_path, dpi=300)
    plt.show()
    print("Saved:", fig_path)
else:
    print("Skipped: incremental performance summary not available.")

## Scientific Interpretation

The manuscript should be framed as a transparent reproducible benchmark. The primary conclusion is that clinical, DaTSCAN/SBR, and SAA-enhanced models achieved only modest held-out discrimination, and multimodal additions did not provide a sufficiently strong or stable incremental improvement to support a deployable clinical prediction claim.

In [ ]:
# ============================================================
# 13. Scientific interpretation table
# ============================================================

interpretation_rows = [
    {
        "topic": "Primary finding",
        "interpretation": (
            "Prediction of rapid motor progression remained challenging, with best held-out discrimination around ROC-AUC ≈ 0.60."
        )
    },
    {
        "topic": "Clinical-only model",
        "interpretation": (
            "Clinical predictors should be retained as the primary baseline comparator because they are simpler and more reproducible."
        )
    },
    {
        "topic": "DaTSCAN/SBR",
        "interpretation": (
            "DaTSCAN/SBR features should be reported as a secondary multimodal analysis because incremental gain was small."
        )
    },
    {
        "topic": "SAA biomarkers",
        "interpretation": (
            "SAA features should be reported as exploratory because they improved some sensitivity/F1 trade-offs but did not consistently improve PR-AUC or specificity."
        )
    },
    {
        "topic": "Clinical use",
        "interpretation": (
            "Models should not be presented as clinically deployable without external validation and stronger predictive performance."
        )
    },
    {
        "topic": "Next manuscript direction",
        "interpretation": (
            "Emphasize reproducibility, transparent limitations, benchmark value, and the difficulty of early prediction using baseline PPMI features."
        )
    }
]

interpretation_table = pd.DataFrame(interpretation_rows)
save_table(interpretation_table, "07_scientific_interpretation_table.csv")
interpretation_table

## Quality Control Checklist

In [ ]:
# ============================================================
# 14. Quality Control Checklist
# ============================================================

qc_rows = []

def qc_pass(condition, item, details_pass="", details_fail=""):
    qc_rows.append({
        "qc_item": item,
        "status": "PASS" if condition else "FAIL",
        "details": details_pass if condition else details_fail
    })

qc_pass(not cohort.empty, "Primary analytic cohort loaded", f"rows={len(cohort)}", "Cohort file missing.")
qc_pass(not final_perf.empty, "Manuscript performance table loaded", f"rows={len(final_perf)}", "Final performance file missing.")
qc_pass(not decision.empty, "Final decision table loaded", f"rows={len(decision)}", "Decision table missing.")
qc_pass(not incremental.empty, "Incremental summary loaded", f"rows={len(incremental)}", "Incremental summary missing.")
qc_pass((FIG_DIR / "figure_01_annualized_delta_NP3TOT_distribution.png").exists() or cohort.empty,
        "Outcome distribution figure saved", "figure generated or skipped due missing cohort.", "Figure missing.")
qc_pass((FIG_DIR / "figure_02_top_models_by_roc_auc.png").exists() or final_perf.empty,
        "Top model ROC-AUC figure saved", "figure generated or skipped due missing final performance.", "Figure missing.")
qc_pass(True, "No new model tuning performed", "Notebook only consolidates previous outputs.", "")

qc = pd.DataFrame(qc_rows)
save_table(qc, "08_quality_control_checklist.csv")
qc

## Expected Output

This notebook saves:

- Tables in `outputs/notebook_13_manuscript_ready_reporting/tables/`
- Figures in `outputs/notebook_13_manuscript_ready_reporting/figures/`
- A text summary report for manuscript drafting.

In [ ]:
# ============================================================
# 15. Summary report
# ============================================================

best_roc_text = "Not available"
best_pr_text = "Not available"

if not final_perf.empty and "roc_auc" in final_perf.columns:
    best_roc = final_perf.dropna(subset=["roc_auc"]).sort_values("roc_auc", ascending=False).head(1)
    if not best_roc.empty:
        r = best_roc.iloc[0]
        best_roc_text = (
            f"{r.get('model', 'model')} | feature_set={r.get('feature_set', 'NA')} | "
            f"ROC-AUC={r.get('roc_auc', np.nan):.4f} | PR-AUC={r.get('pr_auc', np.nan):.4f}"
        )

if not final_perf.empty and "pr_auc" in final_perf.columns:
    best_pr = final_perf.dropna(subset=["pr_auc"]).sort_values("pr_auc", ascending=False).head(1)
    if not best_pr.empty:
        r = best_pr.iloc[0]
        best_pr_text = (
            f"{r.get('model', 'model')} | feature_set={r.get('feature_set', 'NA')} | "
            f"ROC-AUC={r.get('roc_auc', np.nan):.4f} | PR-AUC={r.get('pr_auc', np.nan):.4f}"
        )

report = f'''
Notebook 13 — Manuscript-Ready Exploratory Analysis, Tables, Figures, and Reporting Checklist
==============================================================================================

Project folder:
{PROJECT_DIR}

Output folder:
{OUT_DIR}

Best held-out ROC-AUC model:
{best_roc_text}

Best held-out PR-AUC model:
{best_pr_text}

Main conclusion:
Prediction performance remains modest. The most defensible manuscript framing is a reproducible benchmark comparing clinical, DaTSCAN/SBR, and SAA biomarker feature sets for prediction of rapid motor progression in Parkinson's disease using PPMI data.

Recommended manuscript framing:
- Do not claim clinical deployment readiness.
- Present clinical-only results as the baseline reference.
- Present DaTSCAN/SBR and SAA models as secondary or exploratory.
- Emphasize transparent workflow, leakage control, held-out test evaluation, and limitations.

Files generated:
- 01_table1_cohort_and_outcome_summary.csv
- 02_train_test_split_summary.csv
- 03_feature_set_summary.csv
- 04_manuscript_ready_model_performance_table.csv
- 05_incremental_value_summary.csv
- 06_final_model_selection_decision_table.csv
- 07_scientific_interpretation_table.csv
- 08_quality_control_checklist.csv
- figure_01_annualized_delta_NP3TOT_distribution.png
- figure_02_top_models_by_roc_auc.png
- figure_03_incremental_performance_changes.png

Next step:
Notebook 14 — Manuscript Drafting Package: title, abstract, methods, results tables, figure legends, and limitations.
'''

report_path = OUT_DIR / "09_notebook_13_summary_report.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

print(report)
print("Saved:", report_path)